In [ ]:
"""
04_label_mapping.py
===================
Map YouTube channels to parent music labels.

INPUT  : india_youtube_enriched.csv
OUTPUT : india_youtube_labeled.csv
"""

from __future__ import annotations

import logging
import re
from pathlib import Path

import pandas as pd


# ── Config ────────────────────────────────────────────────────────────────────

INPUT_FILE        = Path("india_youtube_enriched.csv")
OUTPUT_FILE       = Path("india_youtube_labeled.csv")
DISTRIBUTION_FILE = Path("distribution.csv")

FALLBACK_LABEL = "Independent"


# ── Logging ───────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ── Label definitions ─────────────────────────────────────────────────────────
#
# EXACT_MAP   : channel_title  → label  (highest priority, checked first)
# PATTERN_MAP : regex keyword  → label  (checked in order if no exact match)
#
# To add a new label: add its channels to EXACT_MAP and, if needed,
# a keyword to PATTERN_MAP. No changes to map_label() required.

EXACT_MAP: dict[str, str] = {
    # T-Series
    "T-Series":                 "T-Series",
    "T-Series Apna Punjab":     "T-Series",
    "T-Series Tamil":           "T-Series",
    "T-Series Telugu":          "T-Series",
    "T-Series Kannada":         "T-Series",
    "T-Series Malayalam":       "T-Series",
    "T-Series Bhakti Sagar":    "T-Series",
    "Lahari Music | T-Series":  "T-Series",
    "Lahari Music":             "T-Series",
    # Saregama
    "Saregama Music":           "Saregama",
    "Saregama Telugu":          "Saregama",
    "Saregama Tamil":           "Saregama",
    "Saregama Punjabi":         "Saregama",
    "Saregama Hum Bhojpuri":    "Saregama",
    "Saregama Kannada":         "Saregama",
    # Sony
    "SonyMusicIndiaVEVO":       "Sony",
    "Sony Music India":         "Sony",
    # Zee
    "Zee Music Company":        "Zee",
    "Zee Studios":              "Zee",
    # Tips
    "Tips Official":            "Tips",
    "Tips Punjabi":             "Tips",
    # Times Music
    "Times Music":              "Times Music",
    "Times Music Tamil":        "Times Music",
    # Speed Records
    "Speed Records":            "Speed Records",
    # Think Music
    "Think Music India":        "Think Music",
    # Aditya Music
    "Aditya Music":             "Aditya Music",
    "Aditya Music PlayBack":    "Aditya Music",
    # Panorama
    "Panorama Music":           "Panorama",
    # Desi Melodies
    "Desi Melodies":            "Desi Melodies",
    # White Hill
    "White Hill Music":         "White Hill",
    # DRJ Records
    "DRJ Records":              "DRJ",
    # Play DMF
    "Play DMF":                 "Play DMF",
    # YRF
    "YRF":                      "YRF",
    "YRF Music":                "YRF",
    # Dharma
    "Dharma Productions":       "Dharma",
    # South Indian
    "MRT Music":                "MRT",
    "Anand Audio":              "Anand Audio",
    "A2 Music":                 "A2 Music",
    "SVF":                      "SVF",
    "Sun TV":                   "Sun",
    "Sun Music":                "Sun",
    "Sun Pictures":             "Sun",
    "AP International":         "AP International",
    # Regional / Bhojpuri
    "Wave Music":               "Wave Music",
    "Worldwide Records Bhojpuri": "Worldwide Records",
    "GMJ - Global Music Junction": "Global Music Junction",
    # Independent / Digital
    "Merchant Records":         FALLBACK_LABEL,
    "Indie Music Label":        FALLBACK_LABEL,
    "Mass Appeal India":        FALLBACK_LABEL,
    "Unknown":                  FALLBACK_LABEL,
}

# Compiled once at import time. Order matters: first match wins.
_PATTERN_MAP: list[tuple[re.Pattern, str]] = [
    (re.compile(r"t-series",      re.I), "T-Series"),
    (re.compile(r"saregama",      re.I), "Saregama"),
    (re.compile(r"sony",          re.I), "Sony"),
    (re.compile(r"zee",           re.I), "Zee"),
    (re.compile(r"tips",          re.I), "Tips"),
    (re.compile(r"speed\s*records", re.I), "Speed Records"),
    (re.compile(r"times\s*music", re.I), "Times Music"),
    (re.compile(r"think\s*music", re.I), "Think Music"),
    (re.compile(r"aditya\s*music",re.I), "Aditya Music"),
    (re.compile(r"white\s*hill",  re.I), "White Hill"),
    (re.compile(r"\byrf\b",       re.I), "YRF"),
    (re.compile(r"wave\s*music",  re.I), "Wave Music"),
    (re.compile(r"sun\s*music",   re.I), "Sun"),
    (re.compile(r"sun\s*tv",      re.I), "Sun"),
    (re.compile(r"play\s*dmf",    re.I), "Play DMF"),
    (re.compile(r"panorama",      re.I), "Panorama"),
    (re.compile(r"desi\s*melodies",re.I), "Desi Melodies"),
    (re.compile(r"anand\s*audio", re.I), "Anand Audio"),
    (re.compile(r"mrt\s*music",   re.I), "MRT"),
    (re.compile(r"dharma",        re.I), "Dharma"),
]


# ── Mapping logic ─────────────────────────────────────────────────────────────

def map_label(channel: str | float) -> str:
    """
    Resolve a channel title to its parent music label.

    Priority order:
      1. NaN / missing  → FALLBACK_LABEL
      2. Exact match in EXACT_MAP
      3. First regex match in _PATTERN_MAP
      4. FALLBACK_LABEL
    """
    if pd.isna(channel):
        return FALLBACK_LABEL

    channel = str(channel).strip()

    if channel in EXACT_MAP:
        return EXACT_MAP[channel]

    for pattern, label in _PATTERN_MAP:
        if pattern.search(channel):
            return label

    return FALLBACK_LABEL


# ── Vectorised mapping (fast path) ────────────────────────────────────────────

def apply_labels(series: pd.Series) -> pd.Series:
    """
    Map an entire channel_title Series to labels without a Python-level loop.
    Falls back to map_label() only for values not already in EXACT_MAP.
    """
    # Fast exact-match via map; NaN stays NaN for unmatched entries
    result = series.map(EXACT_MAP)

    # For rows not covered by EXACT_MAP, run the full matcher
    unmatched_mask = result.isna() & series.notna()
    if unmatched_mask.any():
        result[unmatched_mask] = series[unmatched_mask].apply(map_label)

    # Fill any remaining NaN (originally missing channel names)
    return result.fillna(FALLBACK_LABEL)


# ── Reporting ─────────────────────────────────────────────────────────────────

def report(df: pd.DataFrame, top_n: int = 30) -> None:
    """Print a quick mapping audit to stdout."""
    total   = len(df)
    labeled = (df["label"] != FALLBACK_LABEL).sum()
    log.info(
        "Coverage: %d / %d rows mapped to a known label (%.1f%%)",
        labeled, total, 100 * labeled / total if total else 0,
    )

    dist = (
        df["label"]
        .value_counts()
        .rename_axis("label")
        .reset_index(name="count")
    )
    dist.to_csv(DISTRIBUTION_FILE, index=False)
    log.info("Distribution saved → %s", DISTRIBUTION_FILE)

    print("\n── Label distribution ──────────────────────────────────────────\n")
    print(dist.head(top_n).to_string(index=False))

    unmapped_channels = (
        df.loc[df["label"] == FALLBACK_LABEL, "channel_title"]
        .dropna()
        .unique()
    )
    if len(unmapped_channels):
        print(f"\n── Unmapped channels ({len(unmapped_channels)}) ──────────────────────────────\n")
        for ch in sorted(unmapped_channels)[:100]:
            print(" ", ch)
    else:
        print("\nAll channels mapped to a known label.")


# ── Main ──────────────────────────────────────────────────────────────────────

def main() -> None:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

    log.info("Loading %s", INPUT_FILE)
    df = pd.read_csv(INPUT_FILE)
    log.info("Shape: %s", df.shape)

    if "channel_title" not in df.columns:
        raise ValueError("Expected a 'channel_title' column in the input CSV.")

    unique_before = df["channel_title"].dropna().nunique()
    log.info("Unique channels: %d", unique_before)

    df["label"] = apply_labels(df["channel_title"])

    report(df)

    df.to_csv(OUTPUT_FILE, index=False)
    log.info("Saved → %s", OUTPUT_FILE)


if __name__ == "__main__":
    main()

01:40:13  INFO      Loading india_youtube_enriched.csv
01:40:14  INFO      Shape: (28000, 21)
01:40:14  INFO      Unique channels: 1013
01:40:14  INFO      Coverage: 12522 / 28000 rows mapped to a known label (44.7%)



── Label distribution ──────────────────────────────────────────

label
Independent          15478
T-Series              4597
Sony                  1509
Saregama              1339
Zee                    870
Tips                   842
Aditya Music           667
YRF                    504
Wave Music             463
Think Music            388
Worldwide Records      246
Speed Records          229
Desi Melodies          223
Play DMF               150
Sun                    130
White Hill             127
Anand Audio             97
MRT                     67
Times Music             65
Panorama                 7
DRJ                      2

── Unmapped channels (947) ──────────────────────────────

    DjAjit Babu Hi Tech
   Altaf Studio
   Anika Entertainment
   Neeraj Sandhya
   Veena Singer
  360 Entertainment Productions
  5911 Records
  7 Sur Entertainment
  7clouds Hindi
  90's Gaane
  90s Filmy Gaane
  A2 Music Official
  AB Bansal Music
  ADT Music
  AFUSIC
  AJ Films
  AJEET ANAND ENT

01:40:15  INFO      Saved → india_youtube_distribution.csv
